In [ ]:
import os
import glob
import time

# ==============================================================================
# 0. ΡΥΘΜΙΣΕΙΣ ΣΤΑΘΕΡΟΤΗΤΑΣ MACOS (Αποφυγή Kernel Died / Crash)
# ==============================================================================
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
# Αν εκτελεστεί σε CPU, αποφυγή υπερφόρτωσης νημάτων
if not torch.backends.mps.is_available() and not torch.cuda.is_available():
    torch.set_num_threads(2)

import torch.nn as nn
from torch.nn import functional as F

# ==============================================================================
# 1. ΕΠΙΤΑΧΥΝΣΗ ΥΛΙΚΟΥ (Hardware Acceleration)
# ==============================================================================
# Έλεγχος για Apple Silicon (M1/M2/M3/M4) -> mps, αλλιώς Nvidia CUDA, αλλιώς CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("🚀 Εντοπίστηκε Apple Silicon GPU (Metal / MPS)! Η εκπαίδευση θα είναι ταχύτατη.")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("🚀 Εντοπίστηκε Nvidia GPU (CUDA)!")
else:
    device = torch.device('cpu')
    print("⚠️ Χρήση CPU (προτείνεται μικρότερο μέγεθος μοντέλου).")

# ==============================================================================
# 2. ΥΠΕΡΠΑΡΑΜΕΤΡΟΙ (Ρύθμιση για ~10.6 Εκατομμύρια Παραμέτρους - Mac Safe)
# ==============================================================================
batch_size = 32        # Μειωμένο από 64 σε 32 για μέγιστη σταθερότητα στη μνήμη του Mac
block_size = 256       # Μέγεθος παραθύρου συμφραζομένων (Context Length σε χαρακτήρες)
n_embd = 384           # Διάσταση διανυσμάτων ενσωμάτωσης (Embedding Dimension)
n_head = 6             # Κεφαλές προσοχής (384 / 6 = 64 διαστάσεις ανά κεφαλή)
n_layer = 6            # Στοίβαγμα 6 επάλληλων Transformer Blocks
dropout = 0.2          # Τεχνική αποφυγής υπερεκπαίδευσης
learning_rate = 3e-4   # Βέλτιστος ρυθμός μάθησης για Transformers
max_iters = 3000       # 3000 βήματα αρκούν για εξαιρετικά αποτελέσματα
eval_interval = 300    # Μέτρηση σφάλματος κάθε 300 βήματα
eval_iters = 25        # Ελαφρύτερη εκτίμηση για αποφυγή spike μνήμης

# ==============================================================================
# 3. ΣΥΓΧΩΝΕΥΣΗ ΚΑΙ ΦΟΡΤΩΣΗ ΒΙΒΛΙΩΝ (1.txt, 2.txt, κλπ.)
# ==============================================================================
# Αναζήτηση όλων των αρχείων .txt στον τρέχοντα φάκελο
book_files = sorted(glob.glob("*.txt"))
if not book_files:
    print("⚠️ Δεν βρέθηκαν αρχεία .txt! Δημιουργία δοκιμαστικού κειμένου...")
    with open("sample.txt", "w", encoding="utf-8") as f:
        f.write("Project Gutenberg sample book text for testing.\n" * 5000)
    book_files = ["sample.txt"]

print(f"📚 Φόρτωση βιβλίων: {book_files}")
raw_text = ""
for filepath in book_files:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        raw_text += f.read() + "\n"

print(f"📊 Συνολικό μέγεθος κειμένου: {len(raw_text):,} χαρακτήρες ({len(raw_text)/1024/1024:.2f} MB)")

# ==============================================================================
# 4. TOKENIZER (Character-Level) & TRAIN/VAL SPLIT
# ==============================================================================
chars = sorted(list(set(raw_text)))
vocab_size = len(chars)
print(f"🔤 Μέγεθος Λεξιλογίου: {vocab_size} μοναδικοί χαρακτήρες")

char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [char_to_id[c] for c in s if c in char_to_id]
decode = lambda l: ''.join([id_to_char[i] for i in l])

# Μετατροπή σε τανυστή (Tensor)
data = torch.tensor(encode(raw_text), dtype=torch.long)

# Διαχωρισμός: 90% για εκπαίδευση (Train), 10% για επαλήθευση (Validation)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    """ Εκτίμηση του loss σε train και validation set χωρίς υπολογισμό παραγώγων """
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    # Καθαρισμός cache Metal αν είμαστε σε Mac
    if device.type == 'mps':
        torch.mps.empty_cache()
    return out

# ==============================================================================
# 5. ΑΡΧΙΤΕΚΤΟΝΙΚΗ TRANSFORMER (Με Dropout & Residuals)
# ==============================================================================
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = (q @ k.transpose(-2, -1)) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(), # Η σύγχρονη συνάρτηση ενεργοποίησης των GPT μοντέλων
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GutenbergTransformer10M(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Αρχικοποίηση βαρών κατά τα πρότυπα του GPT-2
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ==============================================================================
# 6. ΑΡΧΙΚΟΠΟΙΗΣΗ & ΚΑΤΑΜΕΤΡΗΣΗ ΠΑΡΑΜΕΤΡΩΝ
# ==============================================================================
model = GutenbergTransformer10M().to(device)
total_params = sum(p.numel() for p in model.parameters())
print("=" * 60)
print(f"✨ Το μοντέλο κατασκευάστηκε επιτυχώς!")
print(f"📊 Αριθμός Παραμέτρων: {total_params:,} (Κλίμακα 10M)")
print("=" * 60)

# ==============================================================================
# 7. ΕΚΠΑΙΔΕΥΣΗ (Training Loop)
# ==============================================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("\n--- Έναρξη Εκπαίδευσης στα Βιβλία του Gutenberg ---")
start_time = time.time()

for iter in range(max_iters):
    # Περιοδική μέτρηση σφάλματος σε train & validation
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - start_time
        print(f"Βήμα {iter:4d}/{max_iters} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | Χρόνος: {elapsed:.1f}s")

    # Κανονικό εκπαιδευτικό βήμα
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# ==============================================================================
# 8. ΑΠΟΘΗΚΕΥΣΗ ΤΟΥ ΜΟΝΤΕΛΟΥ (Model Weights)
# ==============================================================================
model_path = "gutenberg_10m_model.pt"
torch.save(model.state_dict(), model_path)
print(f"\n💾 Το εκπαιδευμένο μοντέλο αποθηκεύτηκε στο αρχείο: {model_path} ({os.path.getsize(model_path)/1024/1024:.2f} MB)")

# ==============================================================================
# 9. ΔΟΚΙΜΗ ΠΑΡΑΓΩΓΗΣ ΚΕΙΜΕΝΟΥ
# ==============================================================================
print("\n" + "=" * 60)
print("📝 ΔΟΚΙΜΗ ΠΑΡΑΓΩΓΗΣ ΚΕΙΜΕΝΟΥ ΑΠΟ ΤΟ ΜΟΝΤΕΛΟ 10M")
print("=" * 60)

prompts = [
    "It was a dark and stormy night, ",
    "The secret of the old castle was ",
    "Sherlock Holmes sat quietly in his chair, "
]

for p in prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    context = torch.tensor(encode(p), dtype=torch.long, device=device).unsqueeze(0)
    generated = model.generate(context, max_new_tokens=250, temperature=0.75)
    print(decode(generated[0].tolist()))

🚀 Εντοπίστηκε Apple Silicon GPU (Metal / MPS)! Η εκπαίδευση θα είναι ταχύτατη.
📚 Φόρτωση βιβλίων: ['1.txt', '2.txt', '3.txt', '4.txt']
📊 Συνολικό μέγεθος κειμένου: 2,530,749 χαρακτήρες (2.41 MB)
🔤 Μέγεθος Λεξιλογίου: 108 μοναδικοί χαρακτήρες
✨ Το μοντέλο κατασκευάστηκε επιτυχώς!
📊 Αριθμός Παραμέτρων: 10,821,996 (Κλίμακα 10M)

--- Έναρξη Εκπαίδευσης στα Βιβλία του Gutenberg ---
Βήμα    0/3000 | Train Loss: 4.8313 | Val Loss: 4.8366 | Χρόνος: 12.3s
Βήμα  300/3000 | Train Loss: 2.4761 | Val Loss: 2.5891 | Χρόνος: 266.8s
Βήμα  600/3000 | Train Loss: 2.2605 | Val Loss: 2.3918 | Χρόνος: 518.1s
Βήμα  900/3000 | Train Loss: 1.6875 | Val Loss: 1.9051 | Χρόνος: 774.2s
